# W09 — Assignment único semanal (Limpieza avanzada + Quality Gates)

## Setup

In [ ]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path(".").resolve()
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Missing {DB_PATH}. Run W06 pipeline first.")

con = duckdb.connect(str(DB_PATH))

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'","''") + "'"

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing {RAW_CSV}")

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

print("Setup OK —", con.sql("SELECT COUNT(*) AS n_raw FROM raw_ps").fetchone()[0], "filas raw")

## Parte A — Limpieza avanzada

In [ ]:
# TODO 1: method_synonyms(raw_norm, canonical)
# raw_norm = LOWER(TRIM(discoverymethod)) — forma normalizada del raw
# canonical = snake_case definitivo
# 11 filas (todos los métodos únicos del dataset)

con.execute("DROP TABLE IF EXISTS method_synonyms")
con.execute("CREATE TABLE method_synonyms(raw_norm VARCHAR, canonical VARCHAR)")

con.execute("""
INSERT INTO method_synonyms VALUES
    ('transit',                       'transit'),
    ('radial velocity',               'radial_velocity'),
    ('imaging',                       'imaging'),
    ('microlensing',                  'microlensing'),
    ('pulsar timing',                 'pulsar_timing'),
    ('transit timing variations',     'transit_timing_variations'),
    ('eclipse timing variations',     'eclipse_timing_variations'),
    ('orbital brightness modulation', 'orbital_brightness_modulation'),
    ('astrometry',                    'astrometry'),
    ('pulsation timing variations',   'pulsation_timing_variations'),
    ('disk kinematics',               'disk_kinematics')
""")

con.sql("SELECT * FROM method_synonyms").show()

In [ ]:
# TODO 2: silver_planet_v3
# - hostname_canon      = LOWER(TRIM(hostname))
# - discoverymethod_canon = join con method_synonyms por raw_norm + COALESCE fallback
# - disc_year_int       = TRY_CAST(disc_year AS INTEGER)
# - disc_year_bad       = flag booleano cuando TRY_CAST retorna NULL
# - disc_era            = clasificación por década

con.execute("DROP TABLE IF EXISTS silver_planet_v3")

con.execute("""
CREATE TABLE silver_planet_v3 AS
SELECT
    pl_name,
    LOWER(TRIM(hostname))                                                 AS hostname_canon,
    COALESCE(ms.canonical, LOWER(TRIM(raw_ps.discoverymethod)))           AS discoverymethod_canon,
    TRY_CAST(disc_year AS INTEGER)                                        AS disc_year_int,
    (TRY_CAST(disc_year AS INTEGER) IS NULL)                              AS disc_year_bad,
    CASE
        WHEN TRY_CAST(disc_year AS INTEGER) < 2000 THEN 'pre-2000'
        WHEN TRY_CAST(disc_year AS INTEGER) < 2010 THEN '2000s'
        WHEN TRY_CAST(disc_year AS INTEGER) < 2020 THEN '2010s'
        WHEN TRY_CAST(disc_year AS INTEGER) IS NOT NULL THEN '2020s'
        ELSE 'unknown'
    END                                                                   AS disc_era,
    sy_snum, sy_pnum, sy_dist, ra, dec,
    pl_orbper, pl_rade, pl_bmasse, pl_eqt,
    st_teff, st_rad, st_mass
FROM raw_ps
LEFT JOIN method_synonyms ms
    ON LOWER(TRIM(raw_ps.discoverymethod)) = ms.raw_norm
""")

con.sql("SELECT COUNT(*) AS n_rows FROM silver_planet_v3").show()
con.sql("SELECT COUNT(*) AS disc_year_bad FROM silver_planet_v3 WHERE disc_year_bad").show()
con.sql("SELECT discoverymethod_canon, COUNT(*) AS n FROM silver_planet_v3 GROUP BY discoverymethod_canon ORDER BY n DESC").show()
con.sql("SELECT disc_era, COUNT(*) AS n FROM silver_planet_v3 GROUP BY disc_era ORDER BY disc_era").show()

## Parte B — Quality gates

In [ ]:
# TODO 3: quality_events — 4 checks automatizados
# Schema: ts_utc, check_name, status, metric_value, details

from datetime import datetime, timezone

con.execute("DROP TABLE IF EXISTS quality_events")
con.execute("""
CREATE TABLE quality_events(
    ts_utc        TIMESTAMP,
    check_name    VARCHAR,
    status        VARCHAR,
    metric_value  DOUBLE,
    details       VARCHAR
)
""")

ts = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

# CHECK 1 — paridad de filas raw vs silver
n_raw    = con.sql("SELECT COUNT(*) FROM raw_ps").fetchone()[0]
n_silver = con.sql("SELECT COUNT(*) FROM silver_planet_v3").fetchone()[0]
c1_st    = 'PASS' if n_silver == n_raw else 'FAIL'

# CHECK 2 — nulos en hostname_canon
n_null_host = con.sql("SELECT COUNT(*) FROM silver_planet_v3 WHERE hostname_canon IS NULL").fetchone()[0]
c2_st       = 'PASS' if n_null_host == 0 else 'FAIL'

# CHECK 3 — tasa de disc_year_bad (umbral: < 1%)
n_bad  = con.sql("SELECT COUNT(*) FROM silver_planet_v3 WHERE disc_year_bad").fetchone()[0]
c3_val = round(n_bad / n_silver, 6)
c3_st  = 'PASS' if c3_val < 0.01 else 'FAIL'

# CHECK 4 — nulos en discoverymethod_canon
n_unk  = con.sql("SELECT COUNT(*) FROM silver_planet_v3 WHERE discoverymethod_canon IS NULL").fetchone()[0]
c4_st  = 'PASS' if n_unk == 0 else 'FAIL'

rows = [
    (ts, 'row_count_parity',          c1_st, float(n_silver), f'raw={n_raw}, silver={n_silver}'),
    (ts, 'null_hostname_canon',        c2_st, float(n_null_host), f'{n_null_host} nulls en hostname_canon'),
    (ts, 'disc_year_bad_rate',         c3_st, c3_val, f'{n_bad} bad disc_year de {n_silver} ({c3_val:.4%})'),
    (ts, 'null_discoverymethod_canon', c4_st, float(n_unk), f'{n_unk} nulls en discoverymethod_canon'),
]

for r in rows:
    con.execute("INSERT INTO quality_events VALUES (?, ?, ?, ?, ?)", r)

con.sql("SELECT check_name, status, metric_value, details FROM quality_events ORDER BY check_name").show()

## Entregable único semanal (W09)

Entrega:
- `assignments/W09_assignment_student.ipynb` ejecutado
- `docs/w09_report.md` (usar template)
- `docs/w09_quality.md` (usar template)
- 1 entrada en `docs/decisions_log.md` (usar template)